In [1]:
import pandas as pd

In [2]:
covid_df = pd.read_csv("https://raw.githubusercontent.com/CSSEGISandData/COVID-19/refs/heads/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_confirmed_global.csv")
covid_itly = covid_df[covid_df['Country/Region']=='Italy'].drop(['Province/State','Lat','Long'], axis = 1)


In [3]:
covid_itly_df_long = covid_itly.melt(
    id_vars=['Country/Region'],     
    var_name='date',        
    value_name='COVID_cases' 
)

covid_itly_df_long['date'] = pd.to_datetime(covid_itly_df_long['date'])
first_case_date = covid_itly_df_long.loc[covid_itly_df_long['COVID_cases'] > 0, 'date'].min()


/tmp/ipykernel_4346/1383487542.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  covid_itly_df_long['date'] = pd.to_datetime(covid_itly_df_long['date'])


In [4]:
covid_itly_wrkng_df = covid_itly_df_long[covid_itly_df_long['date']>=first_case_date].drop(['Country/Region'],axis = 1)


In [49]:
import torch 
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt 
import sklearn.linear_model as lm

In [7]:
# create lag features
for lag in range(1, 6):
    covid_itly_wrkng_df[f'y_lag{lag}'] = covid_itly_wrkng_df['COVID_cases'].shift(lag)

# drop rows with NaNs (first 5 rows)
df_itly_model = covid_itly_wrkng_df.dropna()

X = df_itly_model[[f'y_lag{lag}' for lag in range(1, 6)]]

print(X)

          y_lag1      y_lag2      y_lag3      y_lag4      y_lag5
14           2.0         2.0         2.0         2.0         2.0
15           2.0         2.0         2.0         2.0         2.0
16           2.0         2.0         2.0         2.0         2.0
17           3.0         2.0         2.0         2.0         2.0
18           3.0         3.0         2.0         2.0         2.0
...          ...         ...         ...         ...         ...
1138  25603510.0  25603510.0  25576852.0  25576852.0  25576852.0
1139  25603510.0  25603510.0  25603510.0  25576852.0  25576852.0
1140  25603510.0  25603510.0  25603510.0  25603510.0  25576852.0
1141  25603510.0  25603510.0  25603510.0  25603510.0  25603510.0
1142  25603510.0  25603510.0  25603510.0  25603510.0  25603510.0

[1129 rows x 5 columns]


In [50]:
x = torch.tensor(X.values, dtype=torch.float32)

y = df_itly_model['COVID_cases']
y = torch.tensor(y.to_numpy(), dtype=torch.float32).view(-1, 1)

In [51]:
from sklearn.preprocessing import StandardScaler

#scaling the data
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(x)
y_scaled = scaler_y.fit_transform(y)

In [63]:
x = torch.tensor(X_scaled, dtype=torch.float32)
y = torch.tensor(y_scaled, dtype=torch.float32)

In [64]:
D_in = 5   # 5 lag features
H = 2
D_out = 1

In [65]:
#question 2a part i
model = torch.nn.Sequential(
    torch.nn.Linear(5, 2),
    torch.nn.ReLU(),
    torch.nn.Linear(2, 1)
)

In [66]:
loss_fn = torch.nn.MSELoss(reduction='sum')

learning_rate = 1e-4
for t in range(1000):
    # Forward pass: compute predicted y by passing x to the model. Module objects
    # override the __call__ operator so you can call them like functions. When
    # doing so you pass a Tensor of input data to the Module and it produces
    # a Tensor of output data.
    y_pred = model(x)

    # Compute and print loss. We pass Tensors containing the predicted and true
    # values of y, and the loss function returns a Tensor containing the
    # loss.
    loss = loss_fn(y_pred, y)
    if t % 100 == 99:
        print(t, loss.item())

    # Zero the gradients before running the backward pass.
    model.zero_grad()

    # Backward pass: compute gradient of the loss with respect to all the learnable
    # parameters of the model. Internally, the parameters of each Module are stored
    # in Tensors with requires_grad=True, so this call will compute gradients for
    # all learnable parameters in the model.
    loss.backward()

    # Update the weights using gradient descent. Each parameter is a Tensor, so
    # we can access its gradients like we did before.
    with torch.no_grad():
        for param in model.parameters():
            param -= learning_rate * param.grad

99 3.583158254623413
199 0.1446133553981781
299 0.11626209318637848
399 0.11022275686264038
499 0.10825082659721375
599 0.10756167769432068
699 0.10706527531147003
799 0.10657097399234772
899 0.10607904195785522
999 0.10558978468179703


In [67]:
#question 2a part i

y_pred_scaled = model(x).detach().numpy()
y_pred = scaler_y.inverse_transform(y_pred_scaled)

MSE = ((y - y_pred.flatten())**2).mean()
RMSE = np.sqrt(MSE)

print("NN (1 layer) MSE:", MSE, "RMSE:", RMSE)

NN (1 layer) MSE: tensor(1.6189e+14) RMSE: tensor(12723659.)


/tmp/ipykernel_4346/216660401.py:6: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  MSE = ((y - y_pred.flatten())**2).mean()
/tmp/ipykernel_4346/216660401.py:7: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  RMSE = np.sqrt(MSE)


In [ ]:
#question 2a part ii
#switched to LeakyReLu to prevent neurons from dying
model2 = torch.nn.Sequential(
    torch.nn.Linear(5, 2),
    torch.nn.LeakyReLU(0.01),
    torch.nn.Linear(2, 2),
    torch.nn.LeakyReLU(0.01),
    torch.nn.Linear(2, 1)
)

In [27]:
loss_fn = torch.nn.MSELoss(reduction='sum')

learning_rate = 1e-4
for t in range(1000):
    # Forward pass: compute predicted y by passing x to the model. Module objects
    # override the __call__ operator so you can call them like functions. When
    # doing so you pass a Tensor of input data to the Module and it produces
    # a Tensor of output data.
    y_pred = model2(x)

    # Compute and print loss. We pass Tensors containing the predicted and true
    # values of y, and the loss function returns a Tensor containing the
    # loss.
    loss = loss_fn(y_pred, y)
    if t % 100 == 99:
        print(t, loss.item())

    # Zero the gradients before running the backward pass.
    model2.zero_grad()

    # Backward pass: compute gradient of the loss with respect to all the learnable
    # parameters of the model. Internally, the parameters of each Module are stored
    # in Tensors with requires_grad=True, so this call will compute gradients for
    # all learnable parameters in the model.
    loss.backward()

    # Update the weights using gradient descent. Each parameter is a Tensor, so
    # we can access its gradients like we did before.
    with torch.no_grad():
        for param in model2.parameters():
            param -= learning_rate * param.grad

99 58.609947204589844
199 44.915252685546875
299 22.34900665283203
399 0.1876266449689865
499 0.12510348856449127
599 0.11696472018957138
699 0.11468765884637833
799 0.11403832584619522
899 0.11360368877649307
999 0.11317168176174164


In [68]:
#question 2a part ii

y_pred_scaled2 = model2(x).detach().numpy()
y_pred2 = scaler_y.inverse_transform(y_pred_scaled2)

MSE2 = ((y - y_pred2.flatten())**2).mean()
RMSE2 = np.sqrt(MSE2)

print("NN (2 layer) MSE:", MSE2, "RMSE:", RMSE2)

NN (2 layer) MSE: tensor(1.6189e+14) RMSE: tensor(12723654.)


/tmp/ipykernel_4346/1431167760.py:6: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  MSE2 = ((y - y_pred2.flatten())**2).mean()
/tmp/ipykernel_4346/1431167760.py:7: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  RMSE2 = np.sqrt(MSE2)


#question 2a part i and ii interpretation
The RMSE of the NN's was over 12 million while the lag linear model was 23000. Assuming everything was done correctly, this would imply an extreme overfitting of the NN's, causing a large error. 

In [28]:
#problem 2b

split_idx = len(df_itly_model) // 2
train = df_itly_model.iloc[:split_idx].copy()
test = df_itly_model.iloc[split_idx:].copy()

In [ ]:
#problem 2b
X_train = train[[f'y_lag{lag}' for lag in range(1, 6)]].dropna()
y_train = train.loc[X_train.index, 'COVID_cases']

X_test = test[[f'y_lag{lag}' for lag in range(1, 6)]].dropna()
y_test = test.loc[X_test.index, 'COVID_cases']

In [39]:
x_train = torch.tensor(X_train.values, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32)

x_test = torch.tensor(X_test.values, dtype=torch.float32)

In [40]:
#problem 2b part i
model2b_1 = torch.nn.Sequential(
    torch.nn.Linear(5, 2),
    torch.nn.ReLU(),
    torch.nn.Linear(2, 1)
)

In [41]:
loss_fn = torch.nn.MSELoss(reduction='sum')

learning_rate = 1e-4
for t in range(1000):
    # Forward pass: compute predicted y by passing x to the model. Module objects
    # override the __call__ operator so you can call them like functions. When
    # doing so you pass a Tensor of input data to the Module and it produces
    # a Tensor of output data.
    y_pred = model2b_1(x)

    # Compute and print loss. We pass Tensors containing the predicted and true
    # values of y, and the loss function returns a Tensor containing the
    # loss.
    loss = loss_fn(y_pred, y)
    if t % 100 == 99:
        print(t, loss.item())

    # Zero the gradients before running the backward pass.
    model2b_1.zero_grad()

    # Backward pass: compute gradient of the loss with respect to all the learnable
    # parameters of the model. Internally, the parameters of each Module are stored
    # in Tensors with requires_grad=True, so this call will compute gradients for
    # all learnable parameters in the model.
    loss.backward()

    # Update the weights using gradient descent. Each parameter is a Tensor, so
    # we can access its gradients like we did before.
    with torch.no_grad():
        for param in model2b_1.parameters():
            param -= learning_rate * param.grad

99 10.571069717407227
199 0.9858930706977844
299 0.37378063797950745
399 0.23594006896018982
499 0.18225699663162231
599 0.15578559041023254
699 0.1413145363330841
799 0.1333225667476654
899 0.12749795615673065
999 0.12383914738893509


In [43]:
y_pred_1 = model2b_1(x_test).detach().numpy()

MSE_1 = ((y_test.values - y_pred_1.flatten())**2).mean()
RMSE_1 = np.sqrt(MSE_1)

print("NN (1 layer) MSE:", MSE_1, "RMSE:", RMSE_1)

NN (1 layer) MSE: 317537236586575.5 RMSE: 17819574.534387052


In [44]:
#problem 2b part ii
model2b_2 = torch.nn.Sequential(
    torch.nn.Linear(5, 2),
    torch.nn.ReLU(),
    torch.nn.Linear(2, 2),
    torch.nn.ReLU(),
    torch.nn.Linear(2, 1)
)

In [45]:
loss_fn = torch.nn.MSELoss(reduction='sum')

learning_rate = 1e-4
for t in range(1000):
    # Forward pass: compute predicted y by passing x to the model. Module objects
    # override the __call__ operator so you can call them like functions. When
    # doing so you pass a Tensor of input data to the Module and it produces
    # a Tensor of output data.
    y_pred = model2b_2(x)

    # Compute and print loss. We pass Tensors containing the predicted and true
    # values of y, and the loss function returns a Tensor containing the
    # loss.
    loss = loss_fn(y_pred, y)
    if t % 100 == 99:
        print(t, loss.item())

    # Zero the gradients before running the backward pass.
    model2b_2.zero_grad()

    # Backward pass: compute gradient of the loss with respect to all the learnable
    # parameters of the model. Internally, the parameters of each Module are stored
    # in Tensors with requires_grad=True, so this call will compute gradients for
    # all learnable parameters in the model.
    loss.backward()

    # Update the weights using gradient descent. Each parameter is a Tensor, so
    # we can access its gradients like we did before.
    with torch.no_grad():
        for param in model2b_2.parameters():
            param -= learning_rate * param.grad

99 12.883996963500977
199 5.437824249267578
299 2.8381924629211426
399 1.3103305101394653
499 0.565617561340332
599 0.25761523842811584
699 0.14603547751903534
799 0.10860218107700348
899 0.09698010236024857
999 0.09337057173252106


In [46]:
y_pred_2 = model2b_2(x_test).detach().numpy()

MSE_2 = ((y_test.values - y_pred_2.flatten())**2).mean()
RMSE_2 = np.sqrt(MSE_2)

print("NN (2 layer) MSE:", MSE_2, "RMSE:", RMSE_2)

NN (2 layer) MSE: 382585165814.35706 RMSE: 618534.6924905321


#part 2b interpretation
using NN with 1 layer, the RMSE was 17.8 million. With 2 layers, the RMSE was better at 618000. 
But the lag linear model in the previous homework got an RMSE of 43,000. Assuming everything was done correctly, then the NN is showing an extreme overfitting of the training data, causing a large error when trying to predict using the testing data. 